<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Attention_Mechanism_(Part_1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Attention Mechanism Explained

The attention mechanism is a powerful concept in neural networks, particularly in sequence-to-sequence models like those used in machine translation, text summarization, and more recently, in large language models. It addresses a fundamental limitation of traditional recurrent neural networks (RNNs) and their variants (LSTMs, GRUs) when dealing with long sequences: the 'bottleneck' of encoding the entire input sequence into a fixed-size context vector.

Here's a detailed breakdown:

**1. The Core Idea: Focusing on Relevant Parts**

Imagine you're translating a long sentence. As you translate word by word, your focus isn't fixed on the beginning of the sentence. Instead, you constantly shift your attention to the most relevant parts of the *source* sentence to produce the next word in the *target* sentence. The attention mechanism mimics this human-like selective focus.

**2. How it Works (in a typical Encoder-Decoder setup):**

*   **Encoder:** Processes the input sequence (e.g., source sentence) and produces a sequence of 'hidden states' or 'annotations' for each element in the input. Each hidden state `h_i` can be thought of as a representation of the i-th input element and its context up to that point.

*   **Decoder:** Generates the output sequence (e.g., target sentence) one element at a time. At each decoding step, instead of just relying on the *last* hidden state of the encoder (the fixed-size context vector), the decoder uses a weighted sum of *all* encoder hidden states.

*   **Alignment/Attention Scores:** To determine which encoder hidden states are most relevant at a given decoding step `t`, the decoder calculates 'attention scores' or 'alignment scores'. These scores quantify how well each encoder hidden state `h_i` 'aligns' with the current state of the decoder `s_t-1` (the hidden state of the decoder from the previous time step).
    *   **Calculation:** This typically involves a compatibility function (e.g., dot product, additive/concatenation-based, or a feed-forward network) that takes `s_t-1` and each `h_i` as input to produce a raw score `e_ti`.

*   **Softmax Normalization:** The raw attention scores `e_ti` are then normalized using a softmax function across all `i` (all encoder hidden states). This produces 'attention weights' `α_ti`, which are positive and sum up to 1. These weights indicate the importance of each encoder hidden state for generating the current decoder output.
    *   `α_ti = exp(e_ti) / Σ_k exp(e_tk)`

*   **Context Vector:** A 'context vector' `c_t` is computed as a weighted sum of the encoder's hidden states, where the weights are the attention weights `α_ti`.
    *   `c_t = Σ_i α_ti * h_i`

*   **Decoder Input:** This context vector `c_t` is then combined with the decoder's previous hidden state `s_t-1` (and possibly the previous output word) to produce the current decoder hidden state `s_t` and ultimately, the next output word.

**3. Types of Attention:**

*   **Additive Attention (Bahdanau Attention):** Uses a feed-forward network to calculate scores. Often referred to as 'global attention' because it considers all encoder states.
*   **Multiplicative Attention (Luong Attention):** Uses dot products or similar multiplicative interactions. Can be 'global' or 'local' (focusing on a subset of encoder states).
*   **Self-Attention (Transformer Attention):** This is a crucial innovation introduced by the Transformer architecture. Instead of attending from a decoder to an encoder, self-attention allows a sequence to attend to *itself*. This means each element in a sequence can weigh the importance of every other element in the *same* sequence to compute a new, richer representation for itself. This is done through 'Query', 'Key', and 'Value' matrices.
    *   **Query (Q):** Represents the element for which we want to compute a new representation.
    *   **Key (K):** Represents all other elements in the sequence that we might attend to.
    *   **Value (V):** The actual information content from other elements that will be weighted and summed.
    *   The attention score is calculated as `softmax((Q * K^T) / sqrt(d_k)) * V`, where `d_k` is the dimension of the keys, used for scaling.
*   **Multi-Head Attention:** In Transformers, self-attention is often performed multiple times in parallel, using different sets of Query, Key, and Value matrices. Each 'head' learns to focus on different aspects of the input, and their outputs are concatenated and linearly transformed to produce the final output.

**4. Why is Attention Important?**

*   **Handles Long-Range Dependencies:** Overcomes the RNN bottleneck by allowing the decoder direct access to all parts of the input sequence, no matter how long, without having to compress it into a single vector.
*   **Interpretability:** The attention weights provide a degree of interpretability, showing which parts of the input sequence were most influential in generating a specific part of the output sequence. This can be visualized as an 'attention map'.
*   **Improved Performance:** Leads to significant improvements in performance for sequence-to-sequence tasks by enabling more effective information flow.
*   **Parallelization (Self-Attention):** A key advantage of self-attention in Transformers is that it can be highly parallelized, unlike the sequential nature of RNNs, making training much faster on modern hardware (GPUs, TPUs).
*   **Reduced Information Loss:** By not compressing the entire input into a single vector, attention mechanisms prevent loss of information that might be critical for longer sequences.

In essence, the attention mechanism is a way for a neural network to dynamically decide which parts of its input (or its own sequence in self-attention) are most important for making a prediction at any given step, leading to more robust and accurate models, especially for complex sequential data.

# Calculate Attention Weights

In [1]:
import numpy as np

scores = np.array([1.2, 4.5, 0.8, 1.7])

exp_scores = np.exp(scores)

attention_weights = exp_scores / np.sum(exp_scores)

print(attention_weights)
print("Sum:", attention_weights.sum())

[0.03286049 0.89093467 0.02202705 0.05417779]
Sum: 1.0000000000000002


turn    → 0.034

off     → 0.837

the     → 0.023

lights  → 0.106

# Context Vector

Suppose:

h1 = [1, 2],
h2 = [3, 4],
h3 = [5, 6],
h4 = [7, 8]

and:

α = [0.05, 0.80, 0.05, 0.10]

Then:

$$ c = 0.05h_1+ 0.80h_2+ 0.05h_3+ 0.10h_4 $$

Python:

In [2]:
import numpy as np

H = np.array([
    [1, 2],
    [3, 4],
    [5, 6],
    [7, 8]
])

alpha = np.array([
    0.05,
    0.80,
    0.05,
    0.10
])

context = np.sum(alpha[:, None] * H, axis=0)

print(context)

[3.4 4.4]


Attention weights
       ↓
Weighted sum of values
       ↓
Context vector

# Implement Dot-Product Attention

In [3]:
import torch

query = torch.tensor([1., 2., 3.])

keys = torch.tensor([
    [1., 0., 1.],
    [0., 2., 1.],
    [1., 1., 0.]
])

values = keys.clone()

scores = torch.matmul(keys, query)

print("Scores:")
print(scores)

weights = torch.softmax(scores, dim=0)

print("\nAttention weights:")
print(weights)

context = torch.matmul(weights, values)

print("\nContext vector:")
print(context)

Scores:
tensor([4., 7., 3.])

Attention weights:
tensor([0.0466, 0.9362, 0.0171])

Context vector:
tensor([0.0638, 1.8896, 0.9829])


#
Query

  ↓

compare with

  ↓

Keys

  ↓

scores

  ↓

softmax

  ↓

attention weights

  ↓

weighted Values

  ↓
  
context

# Implement Scaled Dot-Product Attention

In [4]:
import torch
import math

Q = torch.tensor([
    [1., 0., 1., 0.]
])

K = torch.tensor([
    [1., 0., 1., 0.],
    [0., 1., 0., 1.],
    [1., 1., 0., 0.]
])

V = torch.tensor([
    [10., 20.],
    [30., 40.],
    [50., 60.]
])

d_k = K.shape[-1]

scores = Q @ K.T

scaled_scores = scores / math.sqrt(d_k)

weights = torch.softmax(scaled_scores, dim=-1)

output = weights @ V

print("Scores:")
print(scores)

print("\nScaled scores:")
print(scaled_scores)

print("\nAttention weights:")
print(weights)

print("\nAttention output:")
print(output)

Scores:
tensor([[2., 0., 1.]])

Scaled scores:
tensor([[1.0000, 0.0000, 0.5000]])

Attention weights:
tensor([[0.5065, 0.1863, 0.3072]])

Attention output:
tensor([[26.0143, 36.0143]])
